# Module Population - Preparation et harmonisation

Ce notebook nettoie les donnees Population (Demographie, PDI, Vulnerabilite des menages), harmonise les colonnes et standardise la dimension temporelle.

In [2]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
if not (ROOT / 'data_raw').exists():
    ROOT = ROOT.parent
RAW_ROOT = ROOT / 'data_raw' / 'population'
OUT_ROOT = ROOT / 'data' / 'Population'

GROUPS = {
    'demographie': {
        'raw': RAW_ROOT / 'demographie',
        'out': OUT_ROOT / 'Demographie',
    },
    'pdi': {
        'raw': RAW_ROOT / 'pdi',
        'out': OUT_ROOT / 'PDI',
    },
    'vulnerabilite_menages': {
        'raw': RAW_ROOT / 'vulnerabilite_menages',
        'out': OUT_ROOT / 'Vulnerabilite_menages',
    },
}

PROJECTION_FOLDERS = {
    'projection_regional': RAW_ROOT / 'demographie' / 'projection_regional',
    'projection_provincial': RAW_ROOT / 'demographie' / 'projection_provincial',
}

for g in GROUPS.values():
    g['out'].mkdir(parents=True, exist_ok=True)

for k, g in GROUPS.items():
    print(f'- {k}: {g["raw"]}')

- demographie: /home/malcolmv/Documents/citadel_data-platform/data_raw/population/demographie
- pdi: /home/malcolmv/Documents/citadel_data-platform/data_raw/population/pdi
- vulnerabilite_menages: /home/malcolmv/Documents/citadel_data-platform/data_raw/population/vulnerabilite_menages


In [11]:
FRENCH_MONTHS = {
    1: 'janvier', 2: 'fevrier', 3: 'mars', 4: 'avril', 5: 'mai', 6: 'juin',
    7: 'juillet', 8: 'aout', 9: 'septembre', 10: 'octobre', 11: 'novembre', 12: 'decembre'
}

COLUMN_RENAMES = {
    'indicateurs': 'indicateur',
    'unit': 'unite',
    'value': 'valeur',
    'date': 'date',
    'year': 'annee_source',
    'operation_level': 'niveau_operation',
    'population_type': 'type_population',
    'population_figures': 'effectif_population',
}

DROP_TECH_COLUMNS = {
    'source', 'source_1', 'source_2'
}

CORE_COLUMNS = {
    'indicateur', 'valeur', 'unite', 'date', 'date_source', 'periode_normalisee',
    'granularite_temporelle', 'annee', 'trimestre', 'mois', 'source_onglet',
    'source_fichier', 'domaine'
}

def normalize_columns(columns):
    clean = []
    seen = {}
    for col in columns:
        s = str(col).strip().lower()
        s = s.replace("'", '')
        s = re.sub(r'[^a-z0-9]+', '_', s)
        s = re.sub(r'_+', '_', s).strip('_')
        if s in seen:
            seen[s] += 1
            s = f'{s}_{seen[s]}'
        else:
            seen[s] = 0
        clean.append(s)
    return clean

def normalize_text(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    s = re.sub(r'\s+', ' ', s)
    return s

def parse_french_number(value):
    if pd.isna(value):
        return np.nan
    s = str(value)
    s = s.replace('\u202f', '').replace('\xa0', '').replace(' ', '')
    s = s.replace(',', '.')
    s = s.strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

def parse_time_value(raw):
    if pd.isna(raw):
        return pd.Series({'date_source': np.nan, 'periode_normalisee': np.nan, 'granularite_temporelle': np.nan, 'annee': np.nan, 'trimestre': np.nan, 'mois': np.nan})

    text = normalize_text(raw)

    m_q = re.fullmatch(r'(\d{4})\s*[Qq]\s*([1-4])', text)
    if m_q:
        y, q = int(m_q.group(1)), int(m_q.group(2))
        return pd.Series({'date_source': text, 'periode_normalisee': f'{y} T{q}', 'granularite_temporelle': 'trimestrielle', 'annee': y, 'trimestre': q, 'mois': np.nan})

    m_m = re.fullmatch(r'(\d{4})\s*[Mm]\s*(0?[1-9]|1[0-2])', text)
    if m_m:
        y, m = int(m_m.group(1)), int(m_m.group(2))
        return pd.Series({'date_source': text, 'periode_normalisee': f'{FRENCH_MONTHS[m]} {y}', 'granularite_temporelle': 'mensuelle', 'annee': y, 'trimestre': int((m-1)/3)+1, 'mois': m})

    m_y = re.fullmatch(r'(\d{4})', text)
    if m_y:
        y = int(m_y.group(1))
        return pd.Series({'date_source': text, 'periode_normalisee': str(y), 'granularite_temporelle': 'annuelle', 'annee': y, 'trimestre': np.nan, 'mois': np.nan})

    m_iso = re.fullmatch(r'(\d{4})-(\d{2})-(\d{2})', text)
    if m_iso:
        y, m = int(m_iso.group(1)), int(m_iso.group(2))
        if 1 <= m <= 12:
            return pd.Series({'date_source': text, 'periode_normalisee': f'{FRENCH_MONTHS[m]} {y}', 'granularite_temporelle': 'mensuelle', 'annee': y, 'trimestre': int((m-1)/3)+1, 'mois': m})

    return pd.Series({'date_source': text, 'periode_normalisee': text, 'granularite_temporelle': 'autre', 'annee': np.nan, 'trimestre': np.nan, 'mois': np.nan})

def normalize_time_column(df):
    if 'date' not in df.columns:
        df['date_source'] = np.nan
        df['periode_normalisee'] = np.nan
        df['granularite_temporelle'] = np.nan
        df['annee'] = np.nan
        df['trimestre'] = np.nan
        df['mois'] = np.nan
        return df
    parsed = df['date'].apply(parse_time_value)
    for c in parsed.columns:
        df[c] = parsed[c]
    df['date'] = df['periode_normalisee']
    df['date_source'] = df['periode_normalisee']
    return df

def drop_empty_rows_and_columns(df):
    before_rows = len(df)
    df = df.dropna(how='all').copy()
    dropped_rows = before_rows - len(df)

    empty_cols = [c for c in df.columns if df[c].isna().all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)

    unnamed_cols = [c for c in df.columns if c.startswith('unnamed')]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)

    sparse_cols = []
    for c in df.columns:
        if c in CORE_COLUMNS:
            continue
        if df[c].isna().mean() >= 0.98:
            sparse_cols.append(c)
    if sparse_cols:
        df = df.drop(columns=sparse_cols)

    dropped_cols = sorted(set(empty_cols + unnamed_cols + sparse_cols))
    return df, dropped_rows, dropped_cols

def coalesce_duplicate_columns(df):
    out = pd.DataFrame(index=df.index)
    for c in df.columns:
        if c in out.columns:
            out[c] = out[c].combine_first(df[c])
        else:
            out[c] = df[c]
    return out

def read_population_file(path):
    frames = []
    ext = path.suffix.lower()

    if ext == '.csv':
        if 'fichier_localite_population' in str(path):
            df = parse_localite_population_csv(path)
        else:
            df = pd.read_csv(path, low_memory=False)
        if not df.empty:
            df['source_onglet'] = np.nan
            frames.append(df)
    elif ext in ('.xlsx', '.xls'):
        xls = pd.ExcelFile(path)
        for sh in xls.sheet_names:
            try:
                df = pd.read_excel(path, sheet_name=sh)
            except Exception:
                continue
            if df.empty:
                continue
            df['source_onglet'] = sh
            frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)

def parse_projection_csv_to_long(path, projection_kind):
    raw = pd.read_csv(path, header=None, low_memory=False)
    raw = raw.dropna(axis=0, how='all').dropna(axis=1, how='all')
    if raw.empty or raw.shape[0] < 5 or raw.shape[1] < 4:
        return pd.DataFrame()

    year_row_idx = None
    sex_row_idx = None
    max_scan = min(12, raw.shape[0])

    for ridx in range(max_scan):
        row = raw.iloc[ridx].tolist()
        years = 0
        for v in row:
            y = parse_french_number(v)
            if not pd.isna(y) and 1900 <= y <= 2100:
                years += 1
        if years >= 5:
            year_row_idx = ridx
            break

    if year_row_idx is None:
        return pd.DataFrame()

    for ridx in range(year_row_idx + 1, min(year_row_idx + 5, raw.shape[0])):
        row_text = [normalize_text(v) for v in raw.iloc[ridx].tolist()]
        labels = {str(v).lower() for v in row_text if not pd.isna(v)}
        if {'homme', 'femme', 'ensemble'}.issubset(labels):
            sex_row_idx = ridx
            break

    if sex_row_idx is None:
        return pd.DataFrame()

    year_row = raw.iloc[year_row_idx].tolist()
    sex_row = raw.iloc[sex_row_idx].tolist()

    col_meta = []
    current_year = np.nan
    for col_idx in range(2, raw.shape[1]):
        y = parse_french_number(year_row[col_idx])
        if not pd.isna(y):
            current_year = int(y)

        sx = normalize_text(sex_row[col_idx])
        if pd.isna(current_year) or sx not in {'Homme', 'Femme', 'Ensemble'}:
            continue
        col_meta.append((col_idx, int(current_year), sx.lower()))

    if not col_meta:
        return pd.DataFrame()

    region_name = region_from_projection_filename(path)
    if region_name.strip().upper() == 'BURKINA FASO':
        return pd.DataFrame()

    records = []
    for row_idx in range(sex_row_idx + 1, raw.shape[0]):
        age = normalize_text(raw.iat[row_idx, 1])
        if pd.isna(age):
            continue

        for col_idx, annee, sexe in col_meta:
            valeur = parse_french_number(raw.iat[row_idx, col_idx])
            if pd.isna(valeur):
                continue
            records.append({
                'indicateur': 'population_projettee',
                'age': age,
                'sexe': sexe,
                'annee': annee,
                'date_source': str(annee),
                'date': str(annee),
                'periode_normalisee': str(annee),
                'granularite_temporelle': 'annuelle',
                'valeur': valeur,
                'region': region_name,
                'projection_type': projection_kind,
                'source_fichier': path.name,
            })

    return pd.DataFrame(records)

def parse_localite_population_csv(path):
    raw = pd.read_csv(path, header=None, low_memory=False)
    raw = raw.dropna(axis=1, how='all')
    if raw.empty or raw.shape[0] < 6:
        return pd.DataFrame()

    age_headers = [normalize_text(v) for v in raw.iloc[3, 4:].tolist()]
    value_cols = ['hommes', 'femmes', 'ensemble']
    age_cols = [f'age_{normalize_text(v).lower().replace('-', '_').replace(' ', '_')}' for v in age_headers if not pd.isna(v)]
    columns = ['localite'] + value_cols + age_cols

    data = raw.iloc[5:, : 4 + len(age_cols)].copy()
    data.columns = columns
    data['localite'] = data['localite'].map(normalize_text)
    data = data[data['localite'].notna()].copy()

    for col in value_cols + age_cols:
        if col in data.columns:
            data[col] = data[col].map(parse_french_number)

    # La colonne ensemble sert de valeur principale pour limiter les manquants.
    data['valeur'] = data['ensemble']
    data['indicateur'] = 'population_residente'
    data['annee'] = 2019
    data['date_source'] = '2019'
    data['date'] = '2019'
    data['periode_normalisee'] = '2019'
    data['granularite_temporelle'] = 'annuelle'
    data['source_fichier'] = path.name

    return data

def region_from_projection_filename(path):
    name = path.name
    if ' - ' in name:
        right = name.split(' - ', 1)[1]
        stem = Path(right).stem
    else:
        stem = path.stem
    stem = re.sub(r'\s+', ' ', stem).strip()
    return stem

def merge_projection_folder(folder_path, out_path, projection_kind):
    files = sorted([
        p for p in folder_path.glob('*.csv')
        if not p.name.startswith('.~lock')
    ])

    merged_frames = []
    logs = []

    for fp in files:
        region_name = region_from_projection_filename(fp)
        if region_name.strip().upper() == 'BURKINA FASO':
            # Demande utilisateur: ne pas toucher le fichier BURKINA FASO
            continue

        df = parse_projection_csv_to_long(fp, projection_kind)
        if df.empty:
            continue

        merged_frames.append(df)
        logs.append({
            'fichier': fp.name,
            'region': region_name,
            'lignes': int(len(df)),
        })

    if not merged_frames:
        return None, logs

    out_df = pd.concat(merged_frames, ignore_index=True)
    out_df, _, _ = drop_empty_rows_and_columns(out_df)
    out_df.to_csv(out_path, index=False)
    return out_df, logs

def harmonize_frame(df, source_file, group_name):
    df = df.copy()
    df.columns = normalize_columns(df.columns)
    df = coalesce_duplicate_columns(df)

    for c in df.columns:
        if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == object:
            df[c] = df[c].map(normalize_text)

    df = df.rename(columns=COLUMN_RENAMES)
    df = coalesce_duplicate_columns(df)

    if 'effectif_population' in df.columns and 'valeur' not in df.columns:
        df['valeur'] = pd.to_numeric(df['effectif_population'], errors='coerce')
    elif 'valeur' in df.columns:
        df['valeur'] = pd.to_numeric(df['valeur'], errors='coerce')

    df = normalize_time_column(df)

    extra_drop = [c for c in df.columns if c in DROP_TECH_COLUMNS]
    if extra_drop:
        df = df.drop(columns=extra_drop)

    df, dropped_rows, dropped_empty_cols = drop_empty_rows_and_columns(df)

    df['source_fichier'] = source_file
    df['domaine'] = group_name

    return df, dropped_rows, dropped_empty_cols

def missing_report(df):
    return (df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame('pct_manquants')

In [12]:
action_logs = []
exports = []
tables_harmonisees = {}

for group_name, meta in GROUPS.items():
    raw_path = meta['raw']
    out_path = meta['out']

    files = sorted([p for p in raw_path.rglob('*') if p.suffix.lower() in ('.csv', '.xlsx', '.xls')])

    # Les dossiers de projection sont traites separatement (demande utilisateur).
    if group_name == 'demographie':
        files = [
            p for p in files
            if 'projection_regional' not in p.parts
            and 'projection_provincial' not in p.parts
            and not p.name.startswith('.~lock')
        ]

    frames = []

    for fp in files:
        df_raw = read_population_file(fp)
        if df_raw.empty:
            continue

        df_h, dropped_rows, dropped_cols = harmonize_frame(df_raw, fp.name, group_name)

        file_stub = fp.stem.strip().replace(' ', '_')
        out_file = out_path / f'{file_stub}_nettoye.csv'
        df_h.to_csv(out_file, index=False)

        frames.append(df_h)
        exports.append(str(out_file.relative_to(ROOT)))
        action_logs.append({
            'groupe': group_name,
            'fichier': fp.name,
            'lignes': int(len(df_h)),
            'lignes_vides_supprimees': int(dropped_rows),
            'colonnes': int(df_h.shape[1]),
            'colonnes_vides_supprimees': int(len(dropped_cols)),
        })

    if frames:
        concat_df = pd.concat(frames, ignore_index=True)
        concat_df, _, _ = drop_empty_rows_and_columns(concat_df)

        global_out = out_path / f'{group_name}_harmonise_global.csv'
        concat_df.to_csv(global_out, index=False)
        exports.append(str(global_out.relative_to(ROOT)))

        if 'indicateur' not in concat_df.columns:
            concat_df['indicateur'] = np.nan
        if 'valeur' not in concat_df.columns:
            concat_df['valeur'] = np.nan

        annuel = (
            concat_df.groupby(['annee', 'indicateur'], dropna=False, as_index=False)['valeur']
            .mean()
            .rename(columns={'valeur': 'valeur_moyenne'})
            .sort_values(['annee', 'indicateur'])
        )
        ann_out = out_path / f'{group_name}_fusion_annuelle.csv'
        annuel.to_csv(ann_out, index=False)
        exports.append(str(ann_out.relative_to(ROOT)))

        tables_harmonisees[group_name] = concat_df

# Traitement dedie des projections (sans melange avec les autres donnees demographie)
demo_out = GROUPS['demographie']['out']
for projection_kind, folder_path in PROJECTION_FOLDERS.items():
    merged_path = demo_out / f'{projection_kind}_fusion.csv'
    projection_df, logs = merge_projection_folder(folder_path, merged_path, projection_kind)

    for item in logs:
        action_logs.append({
            'groupe': projection_kind,
            'fichier': item['fichier'],
            'region': item['region'],
            'lignes': item['lignes'],
            'lignes_vides_supprimees': 0,
            'colonnes': None,
            'colonnes_vides_supprimees': None,
        })

    if projection_df is not None:
        exports.append(str(merged_path.relative_to(ROOT)))
        tables_harmonisees[projection_kind] = projection_df

actions_df = pd.DataFrame(action_logs)
actions_df.head(30)

/tmp/ipykernel_58743/1021674976.py:276: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['valeur'] = data['ensemble']
/tmp/ipykernel_58743/1021674976.py:277: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data['indicateur'] = 'population_residente'
/tmp/ipykernel_58743/1021674976.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fram

,groupe,fichier,lignes,lignes_vides_supprimees,colonnes,colonnes_vides_supprimees,region
0,demographie,Densité par région et province des quatres der...,236,0,11.0,3.0,NaN
1,demographie,Evolution de la population par groupes d'âge e...,381,0,12.0,3.0,NaN
2,demographie,Population du Burkina Faso par groupes d'âge e...,1539,0,12.0,3.0,NaN
3,demographie,Population du Burkina Faso par province.csv,1242,0,11.0,3.0,NaN
4,demographie,Population du Burkina Faso par région et selon...,1176,0,12.0,3.0,NaN
5,demographie,Population du Burkina Faso par région.csv,13,0,11.0,3.0,NaN
6,demographie,evolution_des_taux_de_fecondite_par_tranche_ag...,72,0,13.0,3.0,NaN
7,demographie,Fichier des localités du Burkina Faso.xlsx - F...,9838,0,29.0,3.0,NaN
8,demographie,Population résidence par année d'âge selon la ...,82,0,147.0,3.0,NaN
9,demographie,Population résidente par année d'âge selon la ...,82,0,51.0,3.0,NaN


In [5]:
print('Verification des formats temporels normalises')
for g, df in tables_harmonisees.items():
    print(f'\n[{g}]')
    cols = [c for c in ['date_source', 'periode_normalisee', 'granularite_temporelle'] if c in df.columns]
    if cols:
        print(df[cols].drop_duplicates().head(6).to_string(index=False))

Verification des formats temporels normalises

[demographie]
date_source periode_normalisee granularite_temporelle
       1985               1985               annuelle
       1996               1996               annuelle
       2006               2006               annuelle
       2019               2019               annuelle
       1960               1960               annuelle
       1975               1975               annuelle

[pdi]
date_source periode_normalisee granularite_temporelle
       2019               2019               annuelle
       2020               2020               annuelle
       2021               2021               annuelle
       2022               2022               annuelle
       2023               2023               annuelle
       2024               2024               annuelle

[vulnerabilite_menages]
date_source periode_normalisee granularite_temporelle
       2009               2009               annuelle
       2018               2018             

In [6]:
for g, df in tables_harmonisees.items():
    print(f'\n### Qualite - {g}')
    print('Dimensions:', df.shape)
    print(df.head(3).to_string(index=False))
    print('Manquants (%):')
    print(missing_report(df).head(12).to_string())
    if 'valeur' in df.columns:
        print('Description statistique valeur:')
        print(df[['valeur']].describe().to_string())


### Qualite - demographie
Dimensions: (14918, 35)
                                                      indicateur         unite date    valeur date_source periode_normalisee granularite_temporelle  annee                                                       source_fichier     domaine tranche_d_age sexe ages provinces regions localite  hommes  femmes  ensemble  age_0  age_1  age_2  age_3  age_4  age_5  age_6_11  age_12_14  age_15  age_16  age_17  age_18_19  age_20_24  age_25_35  age_36_64  age_65_et_plus
Densité par région et province des quatres derniers recensements Habitants/km2 1985 29.415672        1985               1985               annuelle   1985 Densité par région et province des quatres derniers recensements.csv demographie           NaN  NaN  NaN       NaN     NaN      NaN     NaN     NaN       NaN    NaN    NaN    NaN    NaN    NaN    NaN       NaN        NaN     NaN     NaN     NaN        NaN        NaN        NaN        NaN             NaN
Densité par région et provinc

In [13]:
if tables_harmonisees:
    population_all = pd.concat(list(tables_harmonisees.values()), ignore_index=True)
    population_all, _, _ = drop_empty_rows_and_columns(population_all)
else:
    population_all = pd.DataFrame()

global_path = OUT_ROOT / 'population_harmonise_global.csv'
population_all.to_csv(global_path, index=False)

synthese = {
    'date_generation': pd.Timestamp.now().isoformat(),
    'groupes': list(tables_harmonisees.keys()),
    'nombre_groupes': len(tables_harmonisees),
    'nombre_fichiers_exportes': len(exports),
    'exports': exports,
    'actions': action_logs,
}

synthese_path = OUT_ROOT / 'synthese_preparation_population.json'
synthese_path.write_text(json.dumps(synthese, ensure_ascii=False, indent=2), encoding='utf-8')

print('Export principal:', global_path.relative_to(ROOT))
print('Synthese:', synthese_path.relative_to(ROOT))

Export principal: data/Population/population_harmonise_global.csv
Synthese: data/Population/synthese_preparation_population.json


## Recapitulatif

Ce notebook Population:
- charge les fichiers CSV/XLSX par sous-domaine
- harmonise les noms de colonnes et la dimension temporelle
- supprime les lignes vides et les colonnes completement vides
- genere des sorties nettoyees par fichier
- produit une table harmonisee globale et une fusion annuelle par groupe
- genere une synthese JSON de tracabilite

Traitement specifique des projections demographiques:
- les dossiers `projection_regional` et `projection_provincial` sont traites separatement
- leurs fichiers ne sont pas melanges avec les autres jeux demographie
- chaque dossier est fusionne dans un seul fichier (`projection_regional_fusion.csv` et `projection_provincial_fusion.csv`)
- une colonne `region` est ajoutee a partir du nom du fichier source
- le fichier `BURKINA FASO` est explicitement exclu du traitement